# Full-core Management Science simulation study

This is the single reproduction notebook for the final study. It uses the frozen 30 base instances and the original scientific seeds. The primary weak-core and lexicographic analyses are full-core checks for every organization count. The cap-nine TU and strong-core calculations are supplementary robustness checks.

A clean run is long. Run this notebook without another Gurobi study running at the same time. Every stage is append-only and resumable. Set `RUN_FULL_REPRODUCTION = True` in the setup cell only when intentionally regenerating the source checkpoints. Wall-clock runtimes and the identity of tied optima can vary across machines even with fixed seeds, especially with multithreaded Gurobi.

In [ ]:
from pathlib import Path
import importlib
import json
import platform
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'KEP_functions.py').exists():
    raise FileNotFoundError('Start this notebook from the KEP project directory.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import management_science_simulations as ms
import append_lexicographic_floor_results as lex_floor
import management_science_maxcoal30_robustness as cap30
import management_science_full_core as full_core
for module in (ms, lex_floor, cap30, full_core):
    importlib.reload(module)

RUN_FULL_REPRODUCTION = False
REPRODUCTION_ROOT = Path('results/management_science_full_core/reproduction_work')
CAP4_DIR = REPRODUCTION_ROOT / 'cap04_n5'
CAP9_DIR = REPRODUCTION_ROOT / 'cap09'
CAP30_DIR = REPRODUCTION_ROOT / 'cap30'
REPRO_RETRY_PATH = REPRODUCTION_ROOT / 'retry_checkpoint.jsonl'
CANONICAL_PATH = full_core.CANONICAL_PATH

common = dict(
    instance_dir='instances_large',
    num_base_instances=30,
    instance_selection_seed=20260819,
    master_seed=20260819,
    pool_sizes=(100, 200, 500),
    deltas=(2, 3),
    partition_reps=2,
    partition_var_size=1,
    donor_order_reps=3,
    solver='GUROBI',
    mip_gap=0.0,
    run_heuristic_diagnostics=True,
    run_legacy_lexicographic_donor_search=False,
)
cap4_config = ms.StudyConfig(
    **common, output_dir=str(CAP4_DIR), num_players=(5,),
    max_coal_size=4, time_limit_seconds=300, solver_threads=1,
)
cap9_config = ms.StudyConfig(
    **common, output_dir=str(CAP9_DIR), num_players=(5, 10, 20, 30),
    max_coal_size=9, time_limit_seconds=200, solver_threads=16,
)
cap30_config = cap30.robustness_config(CAP30_DIR)

display(pd.Series({
    'run_full_reproduction': RUN_FULL_REPRODUCTION,
    'python': platform.python_version(),
    'platform': platform.platform(),
    'canonical_output': str(CANONICAL_PATH),
    'frozen_instances_verified': len(full_core.validate_frozen_instances()),
}, name='reproduction setup'))

## Seeded base studies

The cap-four run supplies the mathematically exhaustive five-organization fallback calls. The cap-nine run supplies the primary five- and ten-organization calls and the supplementary TU/strong checks. The obsolete procedure that re-optimized all four lexicographic tiers after each donor is disabled.

In [ ]:
if RUN_FULL_REPRODUCTION:
    display(pd.Series(ms.run_study(cap4_config), name='cap 4, n=5 checkpoint'))
    display(pd.Series(ms.run_study(cap9_config), name='cap 9 checkpoint'))
else:
    print('Base studies not rerun. Set RUN_FULL_REPRODUCTION = True for a clean/resumed reproduction.')

In [ ]:
if RUN_FULL_REPRODUCTION:
    display(pd.Series(
        lex_floor.append_lexicographic_floor_results(CAP4_DIR),
        name='cap 4 floor-preserving stabilization',
    ))
    display(pd.Series(
        lex_floor.append_lexicographic_floor_results(CAP9_DIR),
        name='cap 9 floor-preserving stabilization',
    ))
else:
    print('Cap 4/cap 9 floor searches not rerun.')

## Full weak-core and lexicographic studies for 20 and 30 organizations

In [ ]:
if RUN_FULL_REPRODUCTION:
    display(pd.Series(
        cap30.run_weak_full_core_robustness(cap30_config),
        name='cap 30 weak-core study',
    ))
    display(pd.Series(
        cap30.run_lexicographic_full_core_robustness(cap30_config),
        name='cap 30 lexicographic study',
    ))
else:
    print('Cap 30 studies not rerun.')

## One-file consolidation

This selects only the final scientific procedures, attaches provenance, and writes one standards-compliant JSONL file. Intermediate checkpoints remain resumable but are not analysis inputs after consolidation.

In [ ]:
if RUN_FULL_REPRODUCTION:
    base_summary = full_core.consolidate_completed_results(
        CANONICAL_PATH,
        source_cap4=CAP4_DIR / 'raw_results.jsonl',
        source_cap9=CAP9_DIR / 'raw_results.jsonl',
        source_cap30=CAP30_DIR / 'raw_results.jsonl',
    )
    display(pd.Series(base_summary, name='base canonical data'))
else:
    display(pd.Series(
        full_core.validate_canonical_results(CANONICAL_PATH),
        name='saved canonical data',
    ))

## Automatic long-limit pass for the hard markets

This is the selective retry stage used in the reported data. It rebuilds the same graphs, partitions, donor orders, algorithm seeds, and solver seeds, but permits 1,200 seconds per inner MIP with eight threads. It checkpoints every call and skips completed targets when resumed.

In [ ]:
if RUN_FULL_REPRODUCTION:
    retry_summary = full_core.retry_inconclusive_lexicographic_markets(
        CANONICAL_PATH,
        checkpoint_path=REPRO_RETRY_PATH,
        time_limit_seconds=1200,
        solver_threads=8,
    )
    display(pd.Series(retry_summary, name='long-limit selective retry'))
else:
    print('Long-limit retries not rerun; the saved canonical file already contains them.')

## Strict validation, tables, and figures

In [ ]:
validation = full_core.validate_canonical_results(
    CANONICAL_PATH, require_complete=True
)
expected = {
    'markets': 1440,
    'weak_certified_markets': 1440,
    'weak_donor_free_markets': 1439,
    'weak_assisted_markets': 1,
    'lex_initial_stable': 1369,
    'lex_initial_blocked': 71,
    'lex_initial_unresolved': 0,
    'lex_final_donor_free': 1381,
    'lex_final_assisted': 59,
    'lex_final_unresolved': 0,
}
for key, value in expected.items():
    if validation[key] != value:
        raise AssertionError(f'{key}: expected {value}, found {validation[key]}')
artifacts = full_core.build_all_from_completed_results(
    CANONICAL_PATH, rebuild_canonical=False
)
display(pd.Series(validation, name='strict final validation'))
display(pd.Series(artifacts['tables'], name='analysis table'))
display(pd.Series(artifacts['figures'], name='figure'))